<a href="https://colab.research.google.com/github/athfizh/PemMes26_05_Hafizh/blob/main/JS02/JS02-TugasLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---


# **Deskripsi Tugas**


---

Pada tugas pratikum ini Anda akan menggunakan data "Wisconsin Breast Cancer". Data tersebut terdiri dari 569 data yang digunakan untuk mendiagnonis jenis kanker Malignant (M) dan Benign (B). Tugas Anda adalah,

1.   Pisahkan antara variabel yang dapat digunakan dan variabel yang tidak dapat digunakan.

2.   Lakukan proses encoding pada kolom "diagnosis".

3.   Lakukan proses standardisasi pada semua kolom yang memiliki nilai numerik.

---


# **Langkah 0 - Persiapan Lingkungan**


---

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

Kita butuh:

- pandas → untuk manipulasi data tabular
- LabelEncoder → mengubah data kategorikal (teks) menjadi angka
- StandardScaler → menyamakan skala antar kolom numerik

---


# **Langkah 1 - Memuat Data**


---

In [2]:
df = pd.read_csv('wbc.csv')
print(df.shape)   # (569, 33)
df.head()

(569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


---


# **Langkah 2 - Inspeksi Data**


---

In [3]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

,0
id,0
diagnosis,0
radius_mean,0
texture_mean,0
perimeter_mean,0
area_mean,0
smoothness_mean,0
compactness_mean,0
concavity_mean,0
concave points_mean,0


**Kenapa id tidak boleh dipakai sebagai fitur?**

Karena nilainya cuma nomor identitas administratif (seperti nomor rekam medis), tidak ada hubungan sebab-akibat dengan apakah tumornya ganas atau jinak. Kalau tetap dimasukkan sebagai fitur, model bisa "menghafal" berdasarkan angka acak ini alih-alih benar-benar belajar dari karakteristik selnya (fenomena ini disebut data leakage / noise feature).

---


# **Langkah 3 - Pemisahan Variabel**


---

Jawaban nomor 1:

In [4]:
# Buang kolom yang TIDAK bisa digunakan sebagai fitur
df = df.drop(columns=['id', 'Unnamed: 32'])
print(df.shape)   # (569, 31)

# Pisahkan fitur (X) dan target/label (y)
X = df.drop(columns=['diagnosis'])   # 30 kolom fitur numerik
y = df['diagnosis']                  # kolom target: 'M' atau 'B'

print(X.shape)   # (569, 30)
print(y.shape)   # (569,)

(569, 31)
(569, 30)
(569,)


Penjelasan konsep: dalam machine learning, kita selalu memisahkan data menjadi dua kelompok:

- X (fitur/features) → variabel-variabel yang dipakai model untuk "belajar" dan membuat prediksi (di sini: 30 pengukuran sel seperti radius, tekstur, luas area, dst)
- y (target/label) → variabel yang ingin diprediksi (di sini: apakah kankernya ganas M atau jinak B)

---


# **Langkah 4 - Encoding Kolom diagnosis**


---

Jawaban nomor 2:

In [5]:
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

print(df['diagnosis'].value_counts())

diagnosis
0    357
1    212
Name: count, dtype: int64


**Kenapa perlu encoding?**

Algoritma machine learning bekerja dengan operasi matematis (perkalian, penjumlahan, dsb), sehingga tidak bisa memproses teks 'M' atau 'B' secara langsung. LabelEncoder mengubah nilai kategori menjadi angka secara otomatis dan konsisten secara default, ia mengurutkan berdasarkan alfabet (B sebelum M), sehingga B → 0 dan M → 1.

**Catatan penting untuk konteks medis:**

karena M (kelas yang lebih berbahaya) menjadi 1, ini kebetulan cocok dengan konvensi umum "1 = kondisi positif/berisiko". Tapi selalu cek urutan le.classes_ setiap kali pakai LabelEncoder, karena urutan alfabet tidak selalu sesuai logika yang diharapkan.

---


# **Langkah 5 - Standardisasi Kolom Numerik**


---

Jawaban nomor 3:

In [6]:
std = StandardScaler()
X_scaled = std.fit_transform(X)

# Ubah kembali menjadi DataFrame agar nama kolomnya tetap ada
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled_df.head()

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1.097064,-2.073335,1.269934,0.984375,1.568466,3.283515,2.652874,2.532475,2.217515,2.255747,...,1.886690,-1.359293,2.303601,2.001237,1.307686,2.616665,2.109526,2.296076,2.750622,1.937015
1,1.829821,-0.353632,1.685955,1.908708,-0.826962,-0.487072,-0.023846,0.548144,0.001392,-0.868652,...,1.805927,-0.369203,1.535126,1.890489,-0.375612,-0.430444,-0.146749,1.087084,-0.243890,0.281190
2,1.579888,0.456187,1.566503,1.558884,0.942210,1.052926,1.363478,2.037231,0.939685,-0.398008,...,1.511870,-0.023974,1.347475,1.456285,0.527407,1.082932,0.854974,1.955000,1.152255,0.201391
3,-0.768909,0.253732,-0.592687,-0.764464,3.283553,3.402909,1.915897,1.451707,2.867383,4.910919,...,-0.281464,0.133984,-0.249939,-0.550021,3.394275,3.893397,1.989588,2.175786,6.046041,4.935010
4,1.750297,-1.151816,1.776573,1.826229,0.280372,0.539340,1.371011,1.428493,-0.009560,-0.562450,...,1.298575,-1.466770,1.338539,1.220724,0.220556,-0.313395,0.613179,0.729259,-0.868353,-0.397100


---


# **Langkah 6 - Verifikasi Hasil**


---

In [7]:
# Gabungkan kembali target dan fitur yang sudah diproses
df_final = pd.concat([df['diagnosis'], X_scaled_df], axis=1)
df_final.head()

# Cek: rata-rata harus ≈ 0, std harus = 1
print(X_scaled_df.mean().round(3).head())
print(X_scaled_df.std().round(3).head())

radius_mean       -0.0
texture_mean       0.0
perimeter_mean    -0.0
area_mean         -0.0
smoothness_mean   -0.0
dtype: float64
radius_mean        1.001
texture_mean       1.001
perimeter_mean     1.001
area_mean          1.001
smoothness_mean    1.001
dtype: float64
